**Semi, Anti & Cross Joins — union & unionByName**


| Operation | What it does | Rows Returned |
|-----------|--------------|---------------|
| `left_semi` | Returns only the rows from the left DataFrame that have a matching key in the right DataFrame. Only left DataFrame columns are returned. | Matching left rows only |
| `left_anti` | Returns only the rows from the left DataFrame that do **not** have a matching key in the right DataFrame. Only left DataFrame columns are returned. | Non-matching left rows only |
| `cross` | Returns the Cartesian product of both DataFrames (every row from the left joins with every row from the right). | `Left Rows × Right Rows` |
| `union()` | Combines two DataFrames with the **same schema**. Rows are appended in order (duplicates are kept). | All rows from both DataFrames |
| `unionByName()` | Combines two DataFrames by matching **column names** instead of column positions. Useful when column order differs. | All rows from both DataFrames |

In [ ]:
# Create Spark session
# Hadoop AWS connector allows Spark to communicate with Amazon S3
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Day-14")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a49da1f7-d902-4749-b4ef-a3ce2f79cd96;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 132ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

**Task 1**

Using a left semi join, find all orders placed by SMB segment customers. Only order columns should appear in the result — no customer columns.

In [5]:
from pyspark.sql import functions as F
orders_df.join(customers_df.filter(F.col("segment") == "SMB"),on='customer_id',how='left_semi').\
    show(truncate=False)

+-----------+--------+----------+----------+--------+----------+------------+---------+--------------+-------+
|customer_id|order_id|product_id|order_date|quantity|unit_price|discount_pct|status   |payment_method|region |
+-----------+--------+----------+----------+--------+----------+------------+---------+--------------+-------+
|C002       |O0002   |P005      |2023-01-07|1       |449.99    |0           |Delivered|PayPal        |West   |
|C004       |O0004   |P006      |2023-01-12|2       |89.99     |5           |Delivered|Debit Card    |South  |
|C007       |O0007   |P010      |2023-01-20|2       |109.99    |0           |Delivered|PayPal        |South  |
|C010       |O0010   |P007      |2023-01-28|2       |79.99     |0           |Delivered|Debit Card    |West   |
|C013       |O0013   |P009      |2023-02-08|4       |49.99     |0           |Delivered|PayPal        |West   |
|C016       |O0016   |P002      |2023-02-17|5       |29.99     |0           |Delivered|Debit Card    |Midwest|
|

**Task 2**

Using a left anti join, find all customers who have never placed an order. How many are there in our dataset?

In [ ]:
customers_without_orders = customers_df.join(
    orders_df,
    on="customer_id",
    how="left_anti"
)

customers_without_orders.show(truncate=False)

print("Customers who never placed an order:",
      customers_without_orders.count())

+-----------+----------+---------+-----+----+-----+-------+-----------+-------+
|customer_id|first_name|last_name|email|city|state|country|signup_date|segment|
+-----------+----------+---------+-----+----+-----+-------+-----------+-------+
+-----------+----------+---------+-----+----+-----+-------+-----------+-------+



Customers who never placed an order: 0


**Task 3**

Use crossJoin() to generate all possible combinations of region and payment_method from orders.csv.
 How many combinations are there?

In [9]:
regions_df = orders_df.select("region").distinct()

payment_df = orders_df.select("payment_method").distinct()

combinations_df = regions_df.crossJoin(payment_df)

combinations_df.show(truncate=False)

print("Total combinations:", combinations_df.count())

+-------+--------------+
|region |payment_method|
+-------+--------------+
|Midwest|Credit Card   |
|South  |Credit Card   |
|East   |Credit Card   |
|West   |Credit Card   |
|Midwest|PayPal        |
|South  |PayPal        |
|East   |PayPal        |
|West   |PayPal        |
|Midwest|Debit Card    |
|South  |Debit Card    |
|East   |Debit Card    |
|West   |Debit Card    |
+-------+--------------+



Total combinations: 12


**Task 4**

Filter orders.csv into two DataFrames — Delivered orders and Cancelled orders. Use unionByName() to combine them. Then use .distinct() to confirm no duplicates. Compare row counts.

In [18]:
Delivered_order=orders_df.filter(F.col('status')=="Delivered")
Cancelled_order=orders_df.filter(F.col('status')=='Cancelled')
combined_df=Delivered_order.unionByName(Cancelled_order)
distinct_df=Delivered_order.unionByName(Cancelled_order).distinct()
print("Delivered Orders :", Delivered_order.count())
print("Cancelled Orders :", Cancelled_order.count())
print("After unionByName :", combined_df.count())
print("After distinct() :", distinct_df.count())


Delivered Orders : 74


Cancelled Orders : 4


After unionByName : 78
After distinct() : 78
